# 04 · TF-IDF contra a régua na validação

História E4-H1 do [`docs/backlog.md`](../docs/backlog.md). Os números que saem daqui vão para o [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**Pergunta:** o modelo TF-IDF, que lê o texto do contrato, sugere melhores códigos do que a régua "códigos mais comuns" (E4-T2), que sugere sempre os mesmos 5 sem ler nada?

**Dois termos:**
- **TF-IDF** (*term frequency – inverse document frequency*): transforma cada texto num vetor com um peso por palavra. Uma palavra pesa mais se aparece muito **neste** texto e pouco nos **outros**. Ex.: em "limpeza de escolas", "limpeza" pesa muito (é rara no conjunto e diz muito), "de" pesa quase nada (está em todos os textos).
- **Classificador** (aqui `SGDClassifier` com `log_loss`, uma regressão logística por etiqueta): aprende, a partir desses pesos, a probabilidade de cada etiqueta. As 5 sugestões são as 5 etiquetas mais prováveis.

**Onde corre:** o treino é pesado demais para o portátil, por isso corre num pod da Runpod (E4-S0). O pod escreve dois ficheiros em `metrics/e4_h1/`, que este notebook lê:
- `sugestoes_validacao.parquet`: as 5 sugestões do TF-IDF para cada contrato da validação;
- `treino.json`: informação do treino (N, etiquetas, vocabulário, tempos, memória).

**Critérios do E4-H1 que este notebook verifica:** o top-5 do TF-IDF é maior do que o da régua no código completo e na classe, em todos os contratos **e** nos contratos com texto novo. O teste (novembro e dezembro de 2025) não se abre.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import baselines as bl
import comparison as cmp
import evaluation as ev

DATA = Path.cwd().parent / "data" / "processed"
METRICS = Path.cwd().parent / "metrics" / "e4_h1"

info = json.loads((METRICS / "treino.json").read_text(encoding="utf-8"))
N_MIN = info["n_min"]

treino = pd.read_parquet(DATA / "treino.parquet", columns=["codigo"])
val = pd.read_parquet(DATA / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
print(f"treino: {len(treino):,} exemplos · validação: {len(val):,} contratos · N = {N_MIN}")

treino: 478,384 exemplos · validação: 23,945 contratos · N = 10


## As sugestões do TF-IDF

O ficheiro tem uma linha por contrato da validação: `idcontrato` e `sugestoes` (lista de 5 códigos de 8 dígitos, do mais provável para o menos provável). Antes de avaliar, confirma-se que o ficheiro corresponde à validação: um ficheiro de outro conjunto, ou com contratos em falta, daria números sem sentido **sem dar erro nenhum**.

As sugestões alinham-se pelo `idcontrato`, não pela ordem das linhas. Alinhar pela ordem funcionaria hoje, mas bastava o pod gravar as linhas por outra ordem para cada contrato receber as sugestões de outro. Lição geral: junta-se sempre por uma chave, nunca pela posição.

As verificações usam `raise` e não `assert`, porque o Python ignora os `assert` quando corre com `-O`.

In [2]:
CODIGO = re.compile(r"\d{8}")

sug = pd.read_parquet(METRICS / "sugestoes_validacao.parquet", columns=["idcontrato", "sugestoes"])
sug["idcontrato"] = sug["idcontrato"].astype(str)

repetidos = sug["idcontrato"].duplicated()
if repetidos.any():
    raise ValueError(f"{repetidos.sum():,} contratos com mais de uma linha de sugestões")
em_falta = ~val["idcontrato"].isin(sug["idcontrato"])
if em_falta.any():
    raise ValueError(f"{em_falta.sum():,} contratos da validação sem sugestões")
a_mais = ~sug["idcontrato"].isin(val["idcontrato"])
if a_mais.any():
    raise ValueError(f"{a_mais.sum():,} linhas de sugestões para contratos que não estão na validação")

# o parquet devolve cada lista como um array do numpy; a avaliação quer listas
tfidf = sug.set_index("idcontrato")["sugestoes"].reindex(val["idcontrato"]).map(list)
tfidf.index = val.index

mal = tfidf[tfidf.map(lambda s: len(s) != ev.TOP_K or len(set(s)) != ev.TOP_K
                      or not all(isinstance(c, str) and CODIGO.fullmatch(c) for c in s))]
if len(mal):
    raise ValueError(f"{len(mal):,} contratos sem {ev.TOP_K} códigos distintos de 8 dígitos, ex.: {mal.iloc[0]}")
print(f"{len(tfidf):,} contratos com {ev.TOP_K} sugestões cada · ok")

23,945 contratos com 5 sugestões cada · ok


## A régua

A mesma do E4-T2 (`notebooks/03_regua.ipynb`): as 5 etiquetas de treino mais frequentes, contadas por exemplo de treino depois da regra dos códigos raros, com o **mesmo N** que o TF-IDF usou (lido do `treino.json`). Assim o modelo e a régua escolhem entre as mesmas etiquetas.

In [3]:
top = bl.most_common_labels(treino["codigo"], n_min=N_MIN)
regua = bl.suggest_most_common(top, val.index)
print("régua:", " ".join(top))

régua: 33600000 33140000 33690000 33100000 98300000


## Resultados lado a lado

Uma linha por medida (regra de contagem do E4-T1):
- **codigo:** a sugestão é igual a um dos códigos do contrato (só contratos com código abaixo da divisão);
- **classe:** os mesmos 4 primeiros dígitos (só contratos com código ao nível da classe ou abaixo);
- **divisao:** os mesmos 2 primeiros dígitos (todos os contratos);
- **so_divisao:** os contratos que só têm códigos de divisão, à parte.

Três métricas:
- **top-1:** a 1.ª sugestão acerta. Ex.: o contrato tem 90911200 e o modelo sugere primeiro 90911200;
- **top-5:** alguma das 5 sugestões acerta. É a métrica principal, porque o técnico escolhe entre 5;
- **F1 macro:** calcula-se o F1 (equilíbrio entre "quando sugere X, acerta?" e "quando a resposta é X, sugere X?") para cada etiqueta e faz-se a média, com o mesmo peso para todas. Uma etiqueta rara pesa tanto como uma frequente, por isso a régua, que só sugere 5 etiquetas, fica perto de zero.

Os grupos: **todos** os contratos da validação e só os de **texto novo** (texto que não aparece no treino).

In [4]:
rel = pd.concat({"TF-IDF": ev.report(tfidf, val["codigos"], val["visto_no_treino"]),
                 "régua": ev.report(regua, val["codigos"], val["visto_no_treino"])}, axis=1)
lado = rel[[("TF-IDF", "contratos")]].droplevel(0, axis=1)
for m in ("top1", "top5", "f1_macro"):
    lado[f"{m} TF-IDF"] = rel["TF-IDF", m]
    lado[f"{m} régua"] = rel["régua", m]
fmt = {c: ("{:.4f}" if c.startswith("f1") else "{:.1%}") for c in lado.columns if c != "contratos"}
lado.style.format({"contratos": "{:,}", **fmt})

## Texto já visto contra texto novo

Um contrato da validação tem **texto já visto** se o mesmo texto (normalizado) aparece no treino. Nesses contratos o modelo pode acertar só por ter decorado o par texto → código, sem ter aprendido nada que sirva para textos novos. Ex.: se o texto "Aquisição de gasóleo" está no treino com o código 09134100 (gasóleo), acertar nesse contrato da validação não prova que o modelo saiba ler.

A régua não lê o texto, por isso a diferença entre os dois grupos, na régua, vem só de os grupos terem códigos diferentes (verificação 2 do `03_regua`). No TF-IDF, uma diferença muito maior do que a da régua mostra quanto do resultado vem de decorar. Lição geral: sempre que o teste pode conter exemplos iguais aos do treino, mede-se à parte o que é novo.

In [5]:
visto = val["visto_no_treino"].map({True: "texto já visto", False: "texto novo"})
partes = {}
for grupo in ("texto já visto", "texto novo"):
    m = visto == grupo
    tf = ev.evaluate(tfidf[m], val["codigos"][m])
    rg = ev.evaluate(regua[m], val["codigos"][m])
    partes[grupo] = pd.DataFrame({"contratos": tf["contratos"], "top1 TF-IDF": tf["top1"], "top1 régua": rg["top1"],
                                  "top5 TF-IDF": tf["top5"], "top5 régua": rg["top5"]})
por_visto = pd.concat(partes, names=["grupo", "medida"])
por_visto.style.format({"contratos": "{:,}", **{c: "{:.1%}" for c in por_visto.columns if c != "contratos"}})

## A diferença conta, ou é acaso?

A validação é uma amostra: com outros contratos, os números mudariam um pouco. O **bootstrap emparelhado** do E4-T1 (`src/comparison.py`) mede se a diferença aguenta essa variação: tira 1 000 amostras dos contratos de validação (à sorte, com repetição), **as mesmas para os dois modelos**, e conta em quantas cada um ganha. A diferença conta se um deles ganhar em pelo menos 95% das amostras.

Ex.: se o TF-IDF ganhar em 1 000 de 1 000 amostras, a vantagem não é sorte de ter calhado esta validação; se ganhar em 600, não se pode dizer.

A = TF-IDF, B = régua. Faz-se em todos os contratos e só nos de texto novo (cerca de 40 s cada).

In [6]:
novo = ~val["visto_no_treino"]
boot = pd.concat({
    "todos": cmp.compare(tfidf, regua, val["codigos"]),
    "texto novo": cmp.compare(tfidf[novo], regua[novo], val["codigos"][novo]),
}, names=["grupo"])
cols = ["a", "b", "diferenca", "a_melhor", "b_melhor", "conta"]
boot_top5 = boot.loc[(slice(None), ["codigo", "classe"], "top5"), cols]
boot_top5.style.format({"a": "{:.2%}", "b": "{:.2%}", "diferenca": "{:+.2%}", "a_melhor": "{:.1%}", "b_melhor": "{:.1%}"})

,,,a,b,diferenca,a_melhor,b_melhor,conta
grupo,medida,metrica,,,,,,
todos,codigo,top5,43.19%,26.32%,+16.86%,100.0%,0.0%,True
texto novo,codigo,top5,38.50%,23.72%,+14.78%,100.0%,0.0%,True
todos,classe,top5,37.91%,15.60%,+22.31%,100.0%,0.0%,True
texto novo,classe,top5,34.22%,14.84%,+19.37%,100.0%,0.0%,True


A tabela completa (todas as medidas e métricas), para consulta:

In [7]:
boot[cols].style.format({"a": "{:.4f}", "b": "{:.4f}", "diferenca": "{:+.4f}", "a_melhor": "{:.1%}", "b_melhor": "{:.1%}"})

## Veredicto do E4-H1

O TF-IDF bate a régua num caso se a diferença no top-5 conta (bootstrap) **e** é a favor do TF-IDF. O critério do E4-H1 pede os quatro casos: código e classe, em todos os contratos e no texto novo. Se falhar algum, para-se aqui (aviso do E4 no `backlog.md`): há um erro nos dados, no treino ou na avaliação.

In [8]:
veredicto = pd.DataFrame([
    {"grupo": g, "medida": m,
     "TF-IDF bate a régua": bool(boot.loc[(g, m, "top5"), "conta"]
                                 and boot.loc[(g, m, "top5"), "a_melhor"] > boot.loc[(g, m, "top5"), "b_melhor"])}
    for g in ("todos", "texto novo") for m in ("codigo", "classe")
]).set_index(["grupo", "medida"])
display(veredicto)
print("Critério do E4-H1 (top-5 acima da régua):", "CUMPRIDO" if veredicto.all().all() else "NÃO CUMPRIDO")

TF-IDF bate a régua
grupo      medida                     
todos      codigo                 True
           classe                 True
texto novo codigo                 True
           classe                 True

Critério do E4-H1 (top-5 acima da régua): CUMPRIDO


## Medidas hierárquicas

Dão meio acerto a uma 1.ª sugestão perto da etiqueta (Kiritchenko et al., 2005). Ex.: etiqueta 90911200 e sugestão 90911300 partilham a divisão, o grupo, a classe e a categoria, por isso contam quase como acerto; 45000000 não partilha nada. Só se mostram, não decidem.

In [9]:
hier = pd.concat({"TF-IDF": ev.hierarchical_report(tfidf, val["codigos"], val["visto_no_treino"]),
                  "régua": ev.hierarchical_report(regua, val["codigos"], val["visto_no_treino"])}, names=["modelo"])
hier.style.format("{:.3f}")

## Informação do treino

O que o pod escreveu no `treino.json`: o N, quantas etiquetas e palavras (vocabulário) o modelo tem, quanto tempo e memória gastou.

In [10]:
for chave, valor in info.items():
    print(f"{chave}: {valor}")

data: 2026-10-11T19:09:55
n_min: 10
min_df: 2
treino: {'ficheiro': 'treino.parquet', 'datas': ['2022-01-01', '2025-09-30']}
validacao: {'ficheiro': 'validacao.parquet', 'datas': ['2025-10-01', '2025-10-31'], 'contratos': 23945}
exemplos_usados: 478234
exemplos_saidos: 150
etiquetas: 3000
vocabulario: 402084
n_iter: [6]
convergiu: True
tempo_vetorizar_treinar_s: 453.55
tempo_sugerir_s: 28.3
pico_rss_mb: 9975
nucleos: 192
versoes: {'python': '3.10.12', 'sklearn': '1.7.2', 'numpy': '2.2.6', 'scipy': '1.15.3', 'pandas': '2.3.3'}
coef_bytes: 4825008000


## Como ler

As conclusões vão para o [`qualidade_dados.md`](../docs/qualidade_dados.md), com os números das tabelas acima.